# CDSE openEO backup — smoke test

Run this notebook in the **OpenEO** kernel before relying on the live backup.

The goal is only to verify that the participant can:

- import the openEO Python client;
- authenticate with the CDSE account;
- access the Sentinel-2, WorldCover and Copernicus DEM collections;
- access the cloud-mask, slope and aspect processes.

If this notebook passes, continue with the task-specific openEO fallback notebooks.

In [ ]:
import openeo

print("openEO client:", openeo.client_version())

connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

print("Connected:", connection)

In [ ]:
required_collections = [
    "SENTINEL2_L2A",
    "ESA_WORLDCOVER_10M_2021_V2",
    "COPERNICUS_30",
]

available = set(connection.list_collection_ids())
missing = [c for c in required_collections if c not in available]

if missing:
    raise RuntimeError(f"Missing required openEO collection(s): {missing}")

print("Required collections available:")
for collection_id in required_collections:
    print("  ✓", collection_id)

In [ ]:
required_processes = [
    "to_scl_dilation_mask",
    "aggregate_temporal",
    "aggregate_spatial",
    "slope",
    "aspect",
]

for process_id in required_processes:
    try:
        connection.describe_process(process_id)
        print("  ✓", process_id)
    except Exception as exc:
        raise RuntimeError(
            f"Required process '{process_id}' is unavailable on this backend."
        ) from exc

print("\nOPENEO BACKUP IS READY")

### Interpretation

Passing this notebook means the **CDSE openEO backup is technically available**.

It does not mean the course is offline-ready: openEO still requires CDSE services and internet access.